# Lee-Brickell ISD exploration (Strategy 02: ISD syndrome-decoding prototypes)

Standalone exploration of the Lee-Brickell Information Set Decoder (a generalization of Prange ISD - see `prange_isd_exploration.ipynb` in this same folder) applied to the PEP-to-syndrome-decoding transformation. Predates the main prediction-and-repair framework in `Code/core/`; not actively used by later strategies.


In [ ]:
import os
import sys

# instances_generator.py / LEP_prediction_and_repair_v2.py now live in
# Code/core/ (this notebook was relocated during the Aug 2026 folder
# reorganization), so it needs to be added to sys.path explicitly. Tries
# a few relative depths so this works whether Jupyter's cwd is this
# notebook's own folder or the Code/ root.
for _rel in ['../../core', '../core', 'core']:
    if os.path.isdir(_rel):
        sys.path.insert(0, os.path.abspath(_rel))


# PEP transformation to syndrome decoding: tests with Lee-Brickell Information Set decoder

In [1]:
from sage.coding.information_set_decoder import LinearCodeInformationSetDecoder
from instances_generator import *
import random
import signal
import time
import pandas as pd

## Test PEP transformation to syndrome decoding with Lee-Brickell Information Set decoder

In [2]:
n = 7  # length of the code
k = 3   # dimension of the code
q = 7   # size of the finite field
alpha = 0.01  # probability of flipping 0 to a random element
beta = 0.2   # probability of flipping a random element to 0

# Generate a noisy PEP instance
G1, G2, P, P_noisy = generate_noisy_LCE_instance_CBA(n, k, q, alpha, beta, is_monomial=False)

# Transform the problem to a syndrome decoding problem
H_tilde, vectorP_noisy = transform_problem_to_syndrome_decoding(G1, G2, P_noisy)
H_tilde_1, vectorP_noisy_1 = transform_problem_to_syndrome_decoding_H1(G1, G2, P_noisy)

vectorP = transform_secret_to_single_vector(P)

print("Generator matrix G1:")
print(G1)
print("\nGenerator matrix G2:")
print(G2)
print("\nSecret permutation matrix P:")
print(P)
print("\nNoisy hint for P:")
print(P_noisy)

print("\nParity-check matrix H_tilde:")
print(H_tilde)
print("\nNoisy hint vector for P:")
print(vectorP_noisy)

Generator matrix G1:
[1 0 0 5 5 0 6]
[0 1 0 6 2 6 3]
[0 0 1 6 4 2 2]

Generator matrix G2:
[1 0 0 2 1 1 4]
[0 1 0 5 1 3 6]
[0 0 1 2 6 1 2]

Secret permutation matrix P:
[0 0 1 0 0 0 0]
[0 0 0 1 0 0 0]
[0 0 0 0 0 1 0]
[0 0 0 0 0 0 1]
[0 0 0 0 1 0 0]
[0 1 0 0 0 0 0]
[1 0 0 0 0 0 0]

Noisy hint for P:
[0 0 1 0 0 0 0]
[0 0 0 0 0 0 0]
[0 0 0 0 0 1 0]
[0 0 0 0 0 0 1]
[0 0 0 0 1 0 0]
[0 1 0 0 0 0 0]
[1 0 0 0 0 0 0]

Parity-check matrix H_tilde:
[1 0 0 5 5 0 6 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 5 5 0 6 3 0 0 1 1 0 4]
[0 1 0 6 2 6 3 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 6 2 6 3 0 3 0 4 6 4 2]
[0 0 1 6 4 2 2 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 6 4 2 2 0 0 3 4 5 6 6]
[0 0 0 0 0 0 0 1 0 0 5 5 0 6 0 0 0 0 0 0 0 0 0 0 0 0 0 0 5 0 0 4 4 0 2 1 0 0 5 5 0 6 2 0 0 3 3 0 5]
[0 0 0 0 0 0 0 0 1 0 6 2 6 3 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 5 0 2 3 2 1 0 1 0 6 2 6 3 0 2 0 5 4 5 6]
[0 0 0 0 0 0 0 0 0 1 6 4 2 2 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0

## ISD Lee-Brickell Information Set decoder using SageMath

In [3]:
def lee_brickel_ISD(H, c, w, F, timeout=None):
    """
    Lee-Brickell Information Set Decoder (ISD) implementation using SageMath.

    Parameters:
    H : Matrix
        Parity check matrix of the linear code.
    c : Vector
        Received vector (noisy codeword).
    w : int
        Expected Hamming weight of the error.
    F : FiniteField
        Finite field over which the code is defined.
    timeout : int, optional
        Maximum time allowed for the decoding process in seconds (default is 600).

    Returns:
    e : Vector or None
        Estimated error vector, or None if a timeout occurs.
    """
    # Define the internal handler that raises an exception when the alarm rings
    def timeout_handler(signum, frame):
        raise TimeoutError("The decoding process timed out.")

    # transform the input vector to a Sage vector over the finite field
    c_vector = vector(F, c)

    # create a linear code object from the parity check matrix
    C = codes.from_parity_check_matrix(H)

    # initialize the Lee-Brickell decoder with the code and the expected weight
    chosen_p = min(2, w)
    D2 = C.decoder('InformationSet', w, algorithm='Lee-Brickell', search_size=chosen_p)

    # Register the timeout signal handler
    signal.signal(signal.SIGALRM, timeout_handler)
    # Start the countdown timer (expects seconds as an integer)
    if timeout is not None:
        signal.alarm(int(timeout))

    try:
        # decode the received vector to get the estimated codeword
        estimated_codeword = D2.decode_to_code(c_vector)
        
        # extract the estimated error vector
        e = c_vector - estimated_codeword
        return e

    except TimeoutError:
        return None

    finally:
        # disable the alarm so it doesn't interrupt subsequent lines of code
        signal.alarm(0)

### Single Instance Test

In [5]:
n = 7  # length of the code
k = 3   # dimension of the code
q = 7   # size of the finite field
alpha = 0.01  # probability of flipping 0 to a random element
beta = 0.2   # probability of flipping a random element to 0

# Generate a noisy PEP instance
G1, G2, P, P_noisy = generate_noisy_LCE_instance_CBA(n, k, q, alpha, beta, is_monomial=False)

# Transform the problem to a syndrome decoding problem
H_tilde, vectorP_noisy = transform_problem_to_syndrome_decoding(G1, G2, P_noisy)
H_tilde_1, vectorP_noisy_1 = transform_problem_to_syndrome_decoding_H1(G1, G2, P_noisy)

vectorP = transform_secret_to_single_vector(P)

# Set up the finite field and the received vector for decoding
F = GF(q)
vectorP = vector(F, vectorP)
vectorP_noisy = vector(F, vectorP_noisy)

# TODO: Aproximate the weight of the error vector
vectorE = vectorP_noisy - vectorP
w = sum(1 for x in vectorE if x != 0)

# Decode using the Lee-Brickell ISD algorithm
e = lee_brickel_ISD(H_tilde_1, vectorP_noisy_1, w, F)

print("Success:", vectorE == e)

Success: True


### Test with multiple n and k values

In [20]:
# n_list = [7, 20, 30, 50, 100]
n_list = [50, 100]
# k_list = [3, 10, 15, 25, 50]
k_list = [25, 50]
q = 7   # size of the finite field
alpha = 0.01  # probability of flipping 0 to a random element
beta = 0.2   # probability of flipping a random element to 0
num_trials = 1  # number of trials for each parameter set

In [21]:
all_results = []
for n, k in zip(n_list, k_list):
    for i in range(num_trials):
        print(f"\nTesting with n={n}, k={k}")
        # Generate a noisy PEP instance
        G1, G2, P, P_noisy = generate_noisy_LCE_instance_CBA(n, k, q, alpha, beta, is_monomial=False)

        # Transform the problem to a syndrome decoding problem
        H_tilde, vectorP_noisy = transform_problem_to_syndrome_decoding(G1, G2, P_noisy)
        H_tilde_1, vectorP_noisy_1 = transform_problem_to_syndrome_decoding_H1(G1, G2, P_noisy)
        vectorP = transform_secret_to_single_vector(P)

        # Set up the finite field and the received vector for decoding
        F = GF(q)
        vectorP = vector(F, vectorP)
        vectorP_noisy = vector(F, vectorP_noisy)

        # Decode using the Lee-Brickell ISD algorithm
        start_time = time.time()
        # TODO: Aproximate the weight of the error vector
        vectorE = vectorP_noisy - vectorP
        w = sum(1 for x in vectorE if x != 0)
        
        e2 = lee_brickel_ISD(H_tilde_1, vectorP_noisy_1, w, F)
        end_time2 = time.time()
        print(f"Decoding time e2: {end_time2 - start_time:.2f} seconds")
        e1 = lee_brickel_ISD(H_tilde, vectorP_noisy, w, F)
        end_time1 = time.time()
        if e1 is not None:
            print("Success e1:", vectorP_noisy - vectorP == e1)
        else:
            print("Decoding e1 failed due to timeout.")

        if e2 is not None:
            print("Success e2:", vectorP_noisy - vectorP == e2)
        else:
            print("Decoding e2 failed due to timeout.")
        print(f"Decoding time e1: {end_time1 - start_time:.2f} seconds")
        print(f"Decoding time e2: {end_time2 - start_time:.2f} seconds")

        # write results to a .csv file
        results = {
            "i": i,
            "n": n,
            "k": k,
            "q": q,
            "w": w,
            "alpha": alpha,
            "beta": beta,
            "full_H_tilde_size": (H_tilde.nrows(), H_tilde.ncols()),
            "H1_tilde_size": (H_tilde_1.nrows(), H_tilde_1.ncols()),
            "success_e1": vectorP_noisy - vectorP == e1 if e1 is not None else False,
            "success_e2": vectorP_noisy - vectorP == e2 if e2 is not None else False,
            "decoding_time_seconds_e1": end_time1 - start_time,
            "decoding_time_seconds_e2": end_time2 - start_time,
            "timeout_occurred_e1": e1 is None,
            "timeout_occurred_e2": e2 is None
        }
        all_results.append(results)

        # Save results to a CSV file
        df = pd.DataFrame(all_results)
        df.to_csv("decoding_results_PEP_leebrickel_2.csv", index=False)
        
    


Testing with n=50, k=25


KeyboardInterrupt: 